# Wu 2022 (Charville / Stanford-CRC) — CRC CODEX EDA

Exploratory data analysis of the Charville cohort from Wu et al. 2022 (SPACE-GM),
acquired by CODEX multiplexed imaging.

162 patients, 292 regions, 632,280 cells, 40 protein markers (native Charville panel).

This notebook mirrors `schurch2020_eda.py` section-for-section (both are CRC/CODEX).
Key schema differences from Schurch, per
`context_packages/wu2022_integration_contract.md`:
- **`.X` is publisher z-scored** (per-marker centered near 0, ~67% negative), **not raw
  counts** — the raw-`.X` markdown below is adapted accordingly.
- `layers["exprs_norm"]` = per-marker min-max 0–1 within the cohort (the modeling input).
- No neighborhood / cellular-niche annotation exists for Wu, so the Schurch
  `neighborhood_name` views are replaced with `cell_type` colorings throughout.
- Charville is **RESOLVED** for patients (162) — patient-level grouping is valid here
  (unlike the UPMC/DFCI Wu cohorts where `patient_id == region_id`).

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
_r = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "sp_ml").is_dir() and (p / "notebooks").is_dir())
if str(_r) not in sys.path: sys.path.insert(0, str(_r))

import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq

from sp_ml.data.EDA import (
    FS,
    summarize_metadata, spatial_info, cat_breakdown,
    plot_marker_distributions, register_cmap, order_markers,
    prune_and_eval_graph, representative_samples, graph_celltype_panels,
)

## Inline CFG

The Wu cohorts are not wired into `EDA.py`'s CFG system (see the integration contract),
so we define a small inline `cfg` mirroring `SCHURCH_CFG`'s keys. `sample_col` is the
CODEX region, `patient_col` is the resolved patient, `celltype_col` is the coarse shared
lineage, and `cat_cols` lists the Charville clinical label columns broadcast per region.
`label_col` is set to `alive_or_deceased` (a clean 2-class, no-NaN clinical label) so the
representative-sample / GIF helpers — which group by `cfg["label_col"]` — have something
meaningful to stratify on (Wu has no `group_name`/neighborhood annotation).

In [2]:
# CODEX lateral resolution (same instrument family as Schurch: ~377 nm/px).
CFG = {
    "publication": "Wu et al. 2022 — Charville",
    "technology": "CODEX",
    "disease": "CRC",
    "sample_col": "region_id",
    "patient_col": "patient_id",
    "celltype_col": "cell_type",
    "label_col": "alive_or_deceased",
    "um_per_px": 0.377,
    "size_col": "size",
    "arcsinh_cofactor": None,   # publisher z-scored; no arcsinh applied
    "cat_cols": [
        "region_id", "patient_id", "cell_type", "cell_type_raw",
        "primary_outcome", "recurrence", "alive_or_deceased",
        "type_of_first_recurrence", "grade_differentiation",
    ],
}

## Load Data

In [3]:
DATA_PATH = "../../../data/wu2022/charville.h5ad"

adata_raw = ad.read_h5ad(DATA_PATH)
adata_raw

AnnData object with n_obs × n_vars = 632280 × 40
    obs: 'region_id', 'cell_id', 'cell_type_raw', 'size', 'x', 'y', 'cell_type', 'patient_id', 'primary_outcome', 'recurrence', 'alive_or_deceased', 'type_of_first_recurrence', 'surgery_date', 'first_recurrence_date', 'last_contact_date', 'length_of_disease_free_survival', 'grade_differentiation', 'acquisition_id_visualizer', 'sample_label_visualizer'
    var: 'var_name', 'canonical'
    uns: 'dataset', 'preprocessing', 'region_patient_resolution'
    obsm: 'spatial'
    layers: 'exprs_norm'

In [4]:
# Re-run this cell to restore clean state without restarting kernel
adata = adata_raw.copy()
print("dataset:", adata.uns.get("dataset"))
print("region_patient_resolution:", adata.uns.get("region_patient_resolution"))

dataset: Wu2022 — charville
region_patient_resolution: RESOLVED


## Dataset Overview

In [5]:
summarize_metadata(adata, cfg=CFG)

Cells:   632,280
Markers: 40
Samples: 292

Obs columns (19):
  region_id, cell_id, cell_type_raw, size, x, y, cell_type, patient_id, primary_outcome, recurrence, alive_or_deceased, type_of_first_recurrence, surgery_date, first_recurrence_date, last_contact_date, length_of_disease_free_survival, grade_differentiation, acquisition_id_visualizer, sample_label_visualizer

Obsm keys: ['spatial']

Categorical summaries:
  region_id                 292 unique  ['Charville_c001_v001_r001_reg001', 'Charville_c001_v001_r001_reg002', 'Charville_c001_v001_r001_reg003', 'Charville_c001_v001_r001_reg004', 'Charville_c001_v001_r001_reg007', 'Charville_c001_v001_r001_reg008']...
  patient_id                162 unique  ['1047', '21915', '21919', '21926', '21927', '21931']...


  cell_type                   7 unique  ['B cell', 'Myeloid', 'Other', 'Stroma', 'T cell', 'Tumor']...
  cell_type_raw              15 unique  ['B cell', 'Blood vessel', 'CD4 T cell', 'CD8 T cell', 'Granulocyte', 'Macrophage']...


  primary_outcome             2 unique  ['0.0', '1.0', 'NA']
  recurrence                  2 unique  ['0.0', '1.0', 'NA']


  alive_or_deceased           2 unique  ['Alive', 'Dead']
  type_of_first_recurrence   13 unique  ['DIST RECUR MULTIPLE SITES', 'DIST RECUR, BONE', 'DIST RECUR, LIVER', 'DIST RECUR, LUNG', 'DIST RECUR, LYMPH NODE', 'DIST RECUR, PERITONEUM']...


  grade_differentiation       5 unique  ['1', '2', '3', '4', '9']

Var names: ['CD107a', 'CD117', 'CD11b', 'CD11c', 'CD134', 'CD14', 'CD15', 'CD163', 'CD20', 'CD21', 'CD31', 'CD38', 'CD3e', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD49f', 'CD68', 'CD8', 'CollagenIV', 'DAPI', 'FoxP3', 'Gal3', 'GranzymeB', 'HLA-DR', 'ICOS', 'Ki67', 'PD1', 'PDL1', 'PGP9.5', 'PanCK', 'Podoplanin', 'RORgammaT', 'S100A4', 'Siglec8', 'TFAM', 'TIM3', 'Vimentin', 'aSMA']


In [6]:
spatial_info(adata)

x: [3.0, 1917.0]  range = 1914.0
y: [2.0, 1438.0]  range = 1436.0
Estimated resolution (median NN spacing): 12.65 units


In [7]:
cat_breakdown(adata, cfg=CFG)


────────────────────────────────────────────
region_id  (292 unique)
region_id
Charville_c002_v001_r001_reg184    4038
Charville_c003_v001_r001_reg041    3584
Charville_c002_v001_r001_reg155    3578
Charville_c004_v001_r001_reg110    3496
Charville_c003_v001_r001_reg118    3483
Charville_c002_v001_r001_reg212    3358
Charville_c004_v001_r001_reg146    3292
Charville_c002_v001_r001_reg092    3275
Charville_c004_v001_r001_reg115    3258
Charville_c002_v001_r001_reg057    3234
Charville_c003_v001_r001_reg169    3212
Charville_c004_v001_r001_reg141    3211
Charville_c004_v001_r001_reg173    3207
Charville_c003_v001_r001_reg136    3136
Charville_c002_v001_r001_reg191    3135
Charville_c003_v001_r001_reg146    3130
Charville_c003_v001_r001_reg149    3091
Charville_c001_v001_r001_reg022    3088
Charville_c003_v001_r001_reg141    3065
Charville_c001_v001_r001_reg032    3057
Charville_c001_v001_r001_reg073    3054
Charville_c001_v001_r001_reg066    3041
Charville_c001_v001_r001_reg031    3039


## Single Sample Viewer

Wu has no neighborhood annotation, so cells are colored by `cell_type` (coarse lineage).

In [8]:
_one_region = sorted(adata.obs[CFG["sample_col"]].unique())[0]
sq.pl.spatial_scatter(
    adata[adata.obs[CFG["sample_col"]] == _one_region],
    shape=None,
    color=CFG["celltype_col"],
    size=4,
    figsize=(8, 8),
    dpi=150,
    title=str(_one_region),
)

## Subsample Viewer

8 of 292 regions, colored by `cell_type`, shared palette. (We never plot all 292 regions
— too many to render tractably; see the contract's tractability note.)

In [9]:
_sample_ids = sorted(adata.obs[CFG["sample_col"]].unique())[:8]
_n_cols, _n_rows = 4, 2
_fig, _axes = plt.subplots(_n_rows, _n_cols, figsize=(_n_cols * 5, _n_rows * 5), dpi=150, facecolor="white")
for _ax, _sid in zip(_axes.flatten(), _sample_ids):
    sq.pl.spatial_scatter(adata[adata.obs[CFG["sample_col"]] == _sid], shape=None,
                          color=CFG["celltype_col"], size=4, ax=_ax, title=str(_sid))
plt.tight_layout()
plt.show()

## Analysis Setup — squidpy & scanpy

In [10]:
def _layer_stats(M, name):
    M = M.toarray() if hasattr(M, "toarray") else np.asarray(M)
    v = M[~np.isnan(M)]
    print(f"=== {name} global stats ===")
    print(f"  shape:         {M.shape}")
    print(f"  range:         [{v.min():.4f}, {v.max():.4f}]")
    print(f"  mean / median: {v.mean():.4f} / {np.median(v):.4f}")
    print(f"  pct zero:      {(v == 0).mean() * 100:.1f}%")
    print(f"  pct negative:  {(v < 0).mean() * 100:.1f}%")
    print(f"  pct NaN:       {np.isnan(M).mean() * 100:.1f}%")

### Raw `.X` Evaluation

Unlike Schurch (raw CODEX fluorescence counts, 0–54k), Charville `.X` is **publisher
z-scored** on the native 40-marker panel — per-marker centered near 0 with a substantial
negative fraction (~67% of values < 0, verified in the contract). Distributions are
therefore roughly symmetric around zero rather than right-skewed, and `pct zero` is ~0
(z-scores rarely land exactly on 0). This is already variance-stabilized by the
publisher, which is why the `exprs_norm` recipe below is a plain min-max with no
arcsinh / size-norm / winsorize.

In [11]:
plot_marker_distributions(adata)

In [12]:
_layer_stats(adata.X, ".X (publisher z-scored)")

=== .X (publisher z-scored) global stats ===
  shape:         (632280, 40)
  range:         [-11.4989, 41.4070]


  mean / median: 0.0000 / -0.2321
  pct zero:      0.0%
  pct negative:  65.0%
  pct NaN:       0.0%


### Normalized `exprs_norm` Evaluation

`exprs_norm` is the modeling layer: per-marker **min-max 0–1 within the cohort** applied
on top of the publisher z-scores (no arcsinh, no size norm, no winsorize — the publisher
already variance-stabilized via z-score). Every marker now lives on a common `[0, 1]`
scale — the form used for all downstream analysis.

> **Note (PCA):** `exprs_norm` is min-max `[0, 1]`, **not** z-scored. Markers with
> broader spread therefore carry more weight in PCA. If equal per-marker weighting is
> wanted for the embedding specifically, apply `sc.pp.scale` on top of `exprs_norm` for
> the PCA step only (do not re-normalize for the distribution/marker-profile views).

In [13]:
plot_marker_distributions(adata, layer="exprs_norm")

In [14]:
_layer_stats(adata.layers["exprs_norm"], "exprs_norm")

=== exprs_norm global stats ===
  shape:         (632280, 40)
  range:         [0.0000, 1.0000]


  mean / median: 0.1227 / 0.1095
  pct zero:      0.0%
  pct negative:  0.0%
  pct NaN:       0.0%


### Switch analysis layer → `exprs_norm`

Set `.X` to the normalized layer so all scanpy/squidpy calls below operate on it (raw
z-scored `X` remains recoverable from disk). All-NaN markers (cohort-unmeasured) are
dropped by default so sc/sq calls don't choke on NaN — Charville has none (all 40
markers measured), but the guard is kept for cross-dataset uniformity.

In [15]:
_measured = ~np.all(np.isnan(adata.layers["exprs_norm"]), axis=0)
if (~_measured).any():
    print(f"Dropping {(~_measured).sum()} all-NaN marker(s): {list(adata.var_names[~_measured])}")
adata = adata[:, _measured].copy()
adata.X = adata.layers["exprs_norm"].copy()
# canonical functional ordering → all downstream plots inherit it
adata = adata[:, order_markers(list(adata.var_names))].copy()
print(f"Analysis layer set to exprs_norm — {adata.n_vars} markers, {adata.n_obs} cells.")

Analysis layer set to exprs_norm — 40 markers, 632280 cells.


### Cell type x marker profiles (scanpy)

All views below now run on `exprs_norm` (via `.X`).

In [16]:
sc.pl.dotplot(
    adata,
    var_names=list(adata.var_names),
    groupby=CFG["celltype_col"],
    standard_scale="var",
    figsize=(18, 6),
    dendrogram=True,
)

In [17]:
sc.pl.matrixplot(
    adata,
    var_names=list(adata.var_names),
    groupby=CFG["celltype_col"],
    standard_scale="var",
    figsize=(18, 6),
    dendrogram=True,
)

In [18]:
sc.pl.correlation_matrix(adata, groupby=CFG["celltype_col"], figsize=(10, 8))

In [19]:
_corr = pd.DataFrame(adata.X, columns=adata.var_names).corr()
_fig, _ax = plt.subplots(figsize=(14, 12), dpi=120, facecolor="white")
_im = _ax.imshow(_corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
_ax.set_xticks(range(len(_corr.columns)))
_ax.set_xticklabels(_corr.columns, rotation=90, fontsize=FS["sm"])
_ax.set_yticks(range(len(_corr.index)))
_ax.set_yticklabels(_corr.index, fontsize=FS["sm"])
_fig.colorbar(_im, ax=_ax, fraction=0.03, pad=0.02)
_ax.set_title("Wu Charville — marker x marker correlation (exprs_norm, all cells, no aggregation)", fontsize=FS["md"])
plt.tight_layout()
plt.show()

## Spatial Graph Construction — Delaunay vs KNN

Build two graphs using the biologically-motivated defaults:
- **Delaunay** — parameter-free physical adjacency (~6 neighbors, adaptive)
- **KNN k=10** — fixed-size window

`library_key=CFG["sample_col"]` (`region_id`) is essential: it builds the graph *within*
each CODEX region, so cells from different regions (which share overlapping coordinate
ranges) are never wrongly connected.

In [20]:
# Drop cells with no cluster label so squidpy's nhood_enrichment gets a clean categorical.
adata = adata[adata.obs[CFG["celltype_col"]].notna()].copy()
adata.obs[CFG["celltype_col"]] = adata.obs[CFG["celltype_col"]].astype("category").cat.remove_unused_categories()

# KNN needs > KNN_K cells per library; drop regions too small to graph (protects Delaunay too).
KNN_K = 10
_sizes = adata.obs[CFG["sample_col"]].value_counts()
_small = _sizes.index[_sizes <= KNN_K]
if len(_small):
    print(f"Dropping {len(_small)} region(s) with <= {KNN_K} cells (too small to graph)")
    adata = adata[~adata.obs[CFG["sample_col"]].isin(_small)].copy()
adata.obs[CFG["sample_col"]] = adata.obs[CFG["sample_col"]].astype("category").cat.remove_unused_categories()

sq.gr.spatial_neighbors(adata, library_key=CFG["sample_col"], coord_type="generic",
                        delaunay=True, key_added="delaunay")
sq.gr.spatial_neighbors(adata, library_key=CFG["sample_col"], coord_type="generic",
                        n_neighs=KNN_K, key_added="knn")
print("obsp keys:", [k for k in adata.obsp])

INFO     Creating graph using `generic` coordinates and `None` transform and `292` libraries.                      


INFO     Creating graph using `generic` coordinates and `None` transform and `292` libraries.                      


obsp keys: ['delaunay_connectivities', 'delaunay_distances', 'knn_connectivities', 'knn_distances']


### Prune + evaluate (single reproducible call)

`prune_and_eval_graph` estimates the cell pitch (median per-cell shortest edge), prunes
Delaunay edges beyond `factor`× pitch, writes `obsp["delaunay_pruned_connectivities"]`,
and auto-plots degree + edge-length distributions (Delaunay pre/post-prune, KNN). µm scale
comes from `cfg["um_per_px"]`.

In [21]:
prune_and_eval_graph(adata, cfg=CFG, factor=2.5)

Wu2022 — charville: pitch 19.1px (7.2µm)  prune >47.8px (18.0µm)  removed 13.02% edges  Delaunay degree 5.97 → 5.19  |  KNN degree 10.00


np.float32(47.762432)

### Graph overlay on tissue (squidpy)

Edges over ONE region, cells colored by cell type — Delaunay's adaptive mesh vs KNN's
fixed-degree connectivity.

In [22]:
_one = adata[adata.obs[CFG["sample_col"]] == _one_region].copy()
fig, axes = plt.subplots(1, 2, figsize=(22, 11), dpi=150, facecolor="white")
sq.pl.spatial_scatter(_one, shape=None, color=CFG["celltype_col"],
                      connectivity_key="delaunay_pruned_connectivities",
                      edges_width=0.3, edges_color="#888888",
                      size=40, ax=axes[0], title="Delaunay (pruned)")
sq.pl.spatial_scatter(_one, shape=None, color=CFG["celltype_col"],
                      connectivity_key="knn_connectivities",
                      edges_width=0.3, edges_color="#888888",
                      size=40, ax=axes[1], title="KNN (k=10)")
plt.tight_layout()
plt.show()

### Neighborhood enrichment (squidpy)

Which cell types co-localize beyond chance, using the pruned Delaunay graph. Red =
enriched adjacency; blue = avoidance.

In [23]:
sq.gr.nhood_enrichment(adata, cluster_key=CFG["celltype_col"],
                       connectivity_key="delaunay_pruned", seed=0)
sq.pl.nhood_enrichment(adata, cluster_key=CFG["celltype_col"], figsize=(9, 9))

  0%|          | 0/1000 [00:00<?, ?/s]

## Cell-Type Graph Panels — representative regions

Static companion view: the pruned Delaunay graph for one representative region per
`alive_or_deceased` group (median-sized), cells colored by cell type. The cell-type
legend renders as its own figure. We deliberately use a handful of representative regions
(one per clinical group) rather than all 292 to stay tractable.

In [24]:
_reps, _titles = representative_samples(adata, cfg=CFG, by=CFG["label_col"], method="median")
graph_celltype_panels(adata, _reps, cfg=CFG, titles=_titles)